In [7]:
import numpy as np
from folktables import ACSDataSource, ACSMobility, ACSIncome
from matplotlib import pyplot as plt
import torch
from scipy import stats
from scipy.sparse.linalg import lobpcg
from scipy.linalg import eigh, eig
import pandas as pd
from inFairness.distances import MahalanobisDistances, SquaredEuclideanDistance, LogisticRegSensitiveSubspace
from inFairness.fairalgo import SenSeI
from inFairness.auditor import SenSeIAuditor, SenSRAuditor
from tqdm.auto import tqdm
from utils import *

from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import data

/Users/ceolson/miniforge3/lib/python3.12/site-packages/inFairness/utils/ndcg.py:37: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.vmap` is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use `torch.vmap` instead; see the PyTorch 2.0 release notes and/or the `torch.func` migration guide for more details https://pytorch.org/docs/main/func.migrating.html
  vect_normalized_discounted_cumulative_gain = vmap(
/Users/ceolson/miniforge3/lib/python3.12/site-packages/inFairness/utils/ndcg.py:48: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.vmap` is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use `torch.vmap` instead; see the PyTorch 2.0 release notes and/or the `torch.func` migration guide for more details https://pytorch.org/docs/main/func.migrating.html
  monte_carlo_vect_ndcg = v

In [8]:
r = 3
p = 20
n = 120

In [16]:
# data_source = ACSDataSource(survey_year='2018', horizon='1-Year', survey='person')
# acs_data = data_source.get_data(states=["TX"], download=True)
# features, labels, _ = ACSMobility.df_to_pandas(acs_data)

# chi = 1
# synthetic_covariance_diag = chi * np.ones(p)
# synthetic_covariance_indices = np.random.choice(range(p), size=int(p / 2), replace=False)
# synthetic_covariance_diag[synthetic_covariance_indices] = 1 / chi
# features = pd.DataFrame(np.random.multivariate_normal(np.zeros(p), np.diag(synthetic_covariance_diag), size=n * 2))

from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
statlog_german_credit_data = fetch_ucirepo(id=144) 
  
# data (as pandas dataframes) 
features = statlog_german_credit_data.data.features 
labels = statlog_german_credit_data.data.targets

for col_name in features.columns:
    if(features[col_name].dtype == 'object'):
        features = features.astype({col_name: "category"})
        features[col_name] = features[col_name].cat.codes


In [17]:
results = pd.DataFrame(columns=[
        "Iterate",
        "AA^T to Kstar", 
        "Standard Train Loss", 
        "Fair Train Loss", 
        "Audit Mean", 
        "Audit Std", 
        "Audit Lower Bound",
        "True Audit Mean", 
        "True Audit Std", 
        "True Audit Lower Bound",
        "Worst Ratio", 
        "Worst Ratio True"
    ])


In [18]:
X = clean_data(2 * n, len(features.columns), features, cut_columns=False)
X_train = X[:n]
X_test = X[n:]

Y = labels.head(2 * n)
Y_train = Y[:n]
Y_test = Y[n:]

# p = np.shape(X)[-1]

M, S, y, Astar, Kstar = generate_synthetic_data(n, r, p, X_train)


In [6]:
# with open('numpy_saves.npz', 'wb') as f:
#     np.savez(f, S=S, y=y, Astar=Astar, Kstar=Kstar)

In [19]:
A0 = np.random.normal(0, 1, size=(p, r)) / (np.sqrt(p) * np.sqrt(r)) # initialization(n, r, p, S, X_train, y)


In [20]:
A_iterates = []
A = torch.tensor(A0, requires_grad=True, device="cpu").to(torch.float64)
dists = []


In [21]:
y_tensor = torch.tensor(y, device="cpu")
M_tensor = torch.tensor(M, device="cpu")

for iterate in range(500): # 20 * n):
    loss = L(A, y_tensor, M_tensor)
    loss.backward()
    with torch.no_grad():
        A -= A.grad * 0.1
        print(A.detach().cpu().numpy())
        A_iterates.append(A.detach().cpu().numpy())
        dists.append(np.linalg.norm(A.detach().cpu().numpy() @ A.detach().cpu().numpy().T - Kstar))
        A.grad.zero_()
    with open('A_iterates.npy', 'wb') as f:
        np.save(f, np.array(A_iterates))
    print(iterate, dists[-1])
    

[[ 0.03227908  0.04569832 -0.1176533 ]
 [-0.08447596  0.0966703   0.06499874]
 [ 0.05684221  0.07456084  0.06298512]
 [-0.03926178 -0.06088237  0.07731489]
 [-0.1375023   0.02323469 -0.18959756]
 [-0.00546061 -0.22280938 -0.04106196]
 [ 0.1076179   0.23013058 -0.17274203]
 [ 0.16598764 -0.10178312 -0.03650911]
 [ 0.1517694   0.05842339  0.06116139]
 [ 0.07122393 -0.102325   -0.26698004]
 [-0.01024985  0.05588409  0.04364741]
 [ 0.09005952 -0.17182358  0.177253  ]
 [ 0.09007578 -0.11067034  0.11606937]
 [-0.08464865 -0.05772092 -0.1861123 ]
 [ 0.22487305  0.2099791   0.09294765]
 [-0.04564927 -0.13379451  0.02594883]
 [-0.1956853  -0.01931061  0.01851244]
 [ 0.04790603 -0.04730018  0.16836418]
 [ 0.0294565  -0.00971423 -0.01125625]
 [-0.01507735  0.21310287  0.10521634]]
0 0.8108496323829002
[[ 0.02095761  0.0418146  -0.11739335]
 [-0.0622427   0.09236951  0.06555573]
 [ 0.03967231  0.06712381  0.06335359]
 [-0.03903853 -0.06007517  0.05601906]
 [-0.10963088  0.0304622  -0.16579072]
 [-

KeyboardInterrupt: 

In [10]:
A_iterates

[array([[ 4.29527569e-02, -4.44096331e-02, -1.61915363e-01],
        [ 3.81470280e-02, -4.18972044e-02, -2.38366105e-01],
        [ 6.40608027e-02,  1.61241800e-01, -2.20318983e-01],
        [-9.15992113e-02, -3.99311315e-02,  1.10616000e-01],
        [-1.02030616e-01,  1.17433645e-01, -2.62467055e-02],
        [-9.03013240e-02, -8.27264783e-02, -6.08316397e-02],
        [-1.83662382e-01,  2.68121229e-01, -3.14189951e-01],
        [-1.50135466e-04, -1.11285664e-01,  1.16214590e-01],
        [ 1.63005237e-01,  2.15878954e-01,  1.12388905e-01],
        [-1.79643398e-01,  2.05990599e-01, -1.55665315e-01],
        [-1.52818939e-02, -1.00794481e-01,  9.09845598e-02],
        [ 8.16249393e-02, -1.80627525e-01,  9.49686732e-02],
        [ 1.76112868e-01,  3.14998551e-03,  1.05179437e-01],
        [ 1.46273488e-01, -2.38354849e-01, -4.04503234e-02],
        [-1.21739402e-01,  3.86228739e-02,  1.95072404e-01],
        [ 7.30941297e-02,  1.98427985e-01, -1.14169244e-01],
        [-1.29140609e-02

In [25]:
A_iterates_loaded = []
with open("A_iterates.txt", "r") as f:
    text = f.read()
    for matrix in text.split(",\n"):
        A = np.array([[float(x) for x in row.split(" ")] for row in matrix.split("\n")])
        A_iterates_loaded.append(A)

In [26]:
dists_check = []
for A in A_iterates_loaded:
    dists_check.append(np.linalg.norm(A @ A.T - Kstar))

    

In [32]:
X_train_t = torch.Tensor(np.array(X_train))
Y_train_t = torch.Tensor(np.array(Y_train)) - 1

X_test_t = torch.Tensor(np.array(X_test))
Y_test_t = torch.Tensor(np.array(Y_test)) - 1

train_dataset = TrainDataset(X_train_t, Y_train_t)
test_dataset = TrainDataset(X_test_t, Y_test_t)

train_dl = torch.utils.data.DataLoader(train_dataset, batch_size=8)
test_dl = torch.utils.data.DataLoader(test_dataset, batch_size=8)

network_standard = NeuralNet(p)
optimizer = torch.optim.Adam(network_standard.parameters(), lr=1e-3)
loss_fn = torch.nn.functional.binary_cross_entropy

network_standard.train()

for epoch in range(200):

    for x, y in train_dl:
        optimizer.zero_grad()
        y_pred = network_standard(x)
        loss = loss_fn(y_pred, y)
        loss.backward()
        optimizer.step()
    print(loss)

standard_loss = loss

tensor(0.6831, grad_fn=<BinaryCrossEntropyBackward0>)
tensor(0.6708, grad_fn=<BinaryCrossEntropyBackward0>)
tensor(0.6558, grad_fn=<BinaryCrossEntropyBackward0>)
tensor(0.6351, grad_fn=<BinaryCrossEntropyBackward0>)
tensor(0.6089, grad_fn=<BinaryCrossEntropyBackward0>)
tensor(0.5769, grad_fn=<BinaryCrossEntropyBackward0>)
tensor(0.5380, grad_fn=<BinaryCrossEntropyBackward0>)
tensor(0.4944, grad_fn=<BinaryCrossEntropyBackward0>)
tensor(0.4519, grad_fn=<BinaryCrossEntropyBackward0>)
tensor(0.4138, grad_fn=<BinaryCrossEntropyBackward0>)
tensor(0.3811, grad_fn=<BinaryCrossEntropyBackward0>)
tensor(0.3530, grad_fn=<BinaryCrossEntropyBackward0>)
tensor(0.3294, grad_fn=<BinaryCrossEntropyBackward0>)
tensor(0.3093, grad_fn=<BinaryCrossEntropyBackward0>)
tensor(0.2902, grad_fn=<BinaryCrossEntropyBackward0>)
tensor(0.2718, grad_fn=<BinaryCrossEntropyBackward0>)
tensor(0.2551, grad_fn=<BinaryCrossEntropyBackward0>)
tensor(0.2388, grad_fn=<BinaryCrossEntropyBackward0>)
tensor(0.2233, grad_fn=<Bina

In [34]:
input_metric_true = MahalanobisDistances()
input_metric_true.fit(torch.Tensor(Astar @ Astar.T))

output_metric = SquaredEuclideanDistance()
output_metric.fit(num_dims=1)

rho = 5.0
eps = 0.1
auditor_nsteps = 100
auditor_lr = 0.001

In [ ]:
for A in A_iterates:
    network = NeuralNet(p)
    
    input_metric = MahalanobisDistances()
    input_metric.fit(torch.Tensor(A @ A.T))

    alg = SenSeI(network, input_metric, output_metric, loss_fn, rho, eps, auditor_nsteps, auditor_lr)

    optimizer = torch.optim.Adam(network.parameters(), lr=0.001)

    alg.train()

    for epoch in range(4000):
        for x, y in train_dl:
            optimizer.zero_grad()
            result = alg(x, torch.reshape(y, (-1, 1)))
            result.loss.backward()
            optimizer.step()
        if result.loss < standard_loss:
            print("Stopping")
            break

    fair_loss = result.loss

    auditor = SenSeIAuditor(input_metric, output_metric, auditor_nsteps, auditor_lr)
    auditor_true = SenSeIAuditor(input_metric_true, output_metric, auditor_nsteps, auditor_lr)

    audit = auditor.audit(network, X_test_t, Y_test_t, torch.nn.functional.l1_loss)
    audit_true = auditor_true.audit(network, X_test_t, Y_test_t, torch.nn.functional.l1_loss)

    ratios = []
    for X_1 in X_test_t:
        for X_2 in X_test_t:
            ratios.append((output_metric(network(X_1), network(X_2)) / input_metric(X_1, X_2)).detach().numpy())
    ratios = np.array(ratios)
    worst_ratio = np.max(ratios[~np.isnan(ratios)])

    ratios = []
    for X_1 in X_test_t:
        for X_2 in X_test_t:
            ratios.append((output_metric(network(X_1), network(X_2)) / input_metric_true(X_1, X_2)).detach().numpy())
    ratios = np.array(ratios)
    worst_ratio_true = np.max(ratios[~np.isnan(ratios)])

    print(fair_loss.detach().numpy(), audit, audit_true, worst_ratio, worst_ratio_true)

Stopping
0.0004077894 AuditorResponse(lossratio_mean=1.083026, lossratio_std=0.42454663, lower_bound=1.0192787169204858, threshold=None, pval=None, confidence=None, is_model_fair=None) AuditorResponse(lossratio_mean=1.1344274, lossratio_std=0.798413, lower_bound=1.0145426009966887, threshold=None, pval=None, confidence=None, is_model_fair=None) 28.900967 28.118773
Stopping
0.00037046956 AuditorResponse(lossratio_mean=1.0639858, lossratio_std=0.62675077, lower_bound=0.9698767488724106, threshold=None, pval=None, confidence=None, is_model_fair=None) AuditorResponse(lossratio_mean=1.1049693, lossratio_std=0.55810946, lower_bound=1.0211669474100944, threshold=None, pval=None, confidence=None, is_model_fair=None) 28.964573 26.104923
Stopping
0.0003779874 AuditorResponse(lossratio_mean=1.0708649, lossratio_std=0.4832592, lower_bound=0.9983016592580637, threshold=None, pval=None, confidence=None, is_model_fair=None) AuditorResponse(lossratio_mean=1.170506, lossratio_std=0.49155912, lower_boun

/Users/ceolson/miniforge3/lib/python3.12/site-packages/inFairness/auditor/auditor.py:54: RuntimeWarning: invalid value encountered in divide
  loss_ratio = np.divide(loss_vals_adversarial, loss_vals_original)


0.086643405 AuditorResponse(lossratio_mean=1.5420983, lossratio_std=2.2447538, lower_bound=1.2036264776952046, threshold=None, pval=None, confidence=None, is_model_fair=None) AuditorResponse(lossratio_mean=1.4265409, lossratio_std=1.593481, lower_bound=1.1862702418533113, threshold=None, pval=None, confidence=None, is_model_fair=None) 49.21267 53.292904
Stopping
0.00041828764 AuditorResponse(lossratio_mean=1.0611376, lossratio_std=0.51547825, lower_bound=0.9837364838912206, threshold=None, pval=None, confidence=None, is_model_fair=None) AuditorResponse(lossratio_mean=1.2077782, lossratio_std=0.6614677, lower_bound=1.1084562609356106, threshold=None, pval=None, confidence=None, is_model_fair=None) 29.103724 28.16369
Stopping
0.00039198165 AuditorResponse(lossratio_mean=1.1933011, lossratio_std=0.65502983, lower_bound=1.0949457937174567, threshold=None, pval=None, confidence=None, is_model_fair=None) AuditorResponse(lossratio_mean=1.1250627, lossratio_std=0.5148861, lower_bound=1.0477505